# FNNOperator Training on Google Colab

This notebook sets up and runs FNNOperator training on Google Colab with GPU support.


## Step 1: Enable GPU Runtime

**IMPORTANT**: Before running any cells, make sure GPU is enabled:
1. Go to **Runtime** → **Change runtime type**
2. Set **Hardware accelerator** to **GPU** (T4, V100, or A100)
3. Click **Save**
4. The runtime will restart - this is normal!

Then come back and run the cells below.


In [ ]:
# First, check if GPU is available in the runtime
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ GPU detected in runtime!")
    print(result.stdout)
else:
    print("❌ ERROR: GPU not detected in runtime!")
    print("Please go to Runtime > Change runtime type > Set Hardware accelerator to GPU")
    print("Then restart the runtime and run this cell again.")

# Uninstall any existing PyTorch (CPU or CUDA versions)
!pip uninstall torch torchvision torchaudio -y

# Install PyTorch with CUDA 12.4 support
print("\nInstalling PyTorch with CUDA support...")
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# Install other dependencies
!pip install numpy matplotlib


## Step 2: Verify GPU and CUDA Setup


In [ ]:
import torch

print("=" * 60)
print("GPU DIAGNOSTICS")
print("=" * 60)

# Check PyTorch version
print(f"\nPyTorch version: {torch.__version__}")

# Check CUDA availability
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ CUDA Version: {torch.version.cuda}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"✓ GPU Count: {torch.cuda.device_count()}")
    print("\n✅ SUCCESS: GPU is ready for training!")
else:
    print("\n❌ ERROR: CUDA not available!")
    print("\nTroubleshooting steps:")
    print("1. Check Runtime > Change runtime type > Hardware accelerator = GPU")
    print("2. Restart runtime (Runtime > Restart runtime)")
    print("3. Run Step 1 cell again to reinstall PyTorch with CUDA")
    print("4. If still not working, try Runtime > Factory reset runtime")
    
    # Check if it's CPU-only PyTorch
    if "+cpu" in torch.__version__:
        print("\n⚠️ Detected CPU-only PyTorch!")
        print("Run Step 1 cell again to install CUDA version.")


## Step 3: Clone Your Repository (Recommended)

**Option A: Clone from GitHub** (see cell below) - This is the easiest way!

**Option B: Upload files manually** (see Step 4) - Use if repo is not on GitHub


In [ ]:
# Option A: Clone from GitHub
# Replace 'YOUR_USERNAME' and 'YOUR_REPO_NAME' with your actual GitHub details
# Example: !git clone https://github.com/yourusername/FNO_Project.git

# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git

# If your repo is private, you'll need to authenticate:
# Option 1: Use personal access token (recommended for private repos)
# !git clone https://YOUR_TOKEN@github.com/YOUR_USERNAME/YOUR_REPO_NAME.git

# Option 2: Use SSH (requires setting up SSH keys in Colab)
# !git clone git@github.com:YOUR_USERNAME/YOUR_REPO_NAME.git

# After cloning, the repo will be in /content/YOUR_REPO_NAME/
# All your files will be there, including fnnoperator/ folder

print("⚠️ Edit the command above with your GitHub username and repo name, then run this cell!")


In [ ]:
# Option B: Create directory structure (only if NOT cloning from GitHub)
import os
from pathlib import Path

# Create directory structure
os.makedirs("fnnoperator", exist_ok=True)
os.makedirs("fnnoperator/data", exist_ok=True)
os.makedirs("fnnoperator/outputs", exist_ok=True)

print("✓ Directory structure created (skip this if you cloned the repo)")


## Step 4: Upload Code Files (Only if NOT using Git clone)

**Skip this step if you cloned your repository!**

If you didn't clone from GitHub, upload these files manually:
- `FNN.py`
- `FNNOperator.py`
- `FNNTrainer.py`
- `train_heat_operator.py`
- `inference_heat_operator.py` (optional)


In [ ]:
from google.colab import files
import shutil

print("Upload your Python files. Select multiple files by holding Ctrl/Cmd.")
uploaded = files.upload()

# Move uploaded files to fnnoperator directory
for filename in uploaded.keys():
    if filename.endswith('.py'):
        shutil.move(filename, f"fnnoperator/{filename}")
        print(f"✓ Moved {filename} to fnnoperator/")
    else:
        print(f"⚠ Skipped {filename} (not a .py file)")


## Step 5: Prepare Data

**Option A: Upload existing data** (if you have train.npz, test.npz, validate.npz)

**Option B: Generate data in Colab** (see next cell)


In [ ]:
from google.colab import files
import shutil

print("Upload your data files (train.npz, test.npz, validate.npz)")
print("Create a subfolder first, e.g., fnnoperator/data/32x32/")

data_dir = "fnnoperator/data/32x32"
os.makedirs(data_dir, exist_ok=True)

uploaded = files.upload()
for filename in uploaded.keys():
    if filename.endswith('.npz'):
        shutil.move(filename, f"{data_dir}/{filename}")
        print(f"✓ Moved {filename} to {data_dir}/")


## Step 6: Run Training

Now you can run training! Adjust parameters as needed.


In [ ]:
import sys
import os

# Determine the base directory
# If you cloned the repo, it will be in /content/REPO_NAME/
# If you uploaded files manually, use /content/

repo_name = "FNO_Project"  # ⚠️ CHANGE THIS to your actual repo name if you cloned from GitHub

if os.path.exists(f"/content/{repo_name}"):
    # Repository was cloned
    base_dir = f"/content/{repo_name}"
    sys.path.insert(0, base_dir)
    os.chdir(base_dir)
    data_path = f"{repo_name}/fnnoperator/data/32x32"
    print(f"✓ Using cloned repository: {base_dir}")
else:
    # Files were uploaded manually
    base_dir = "/content"
    sys.path.insert(0, base_dir)
    os.chdir(base_dir)
    data_path = "fnnoperator/data/32x32"
    print(f"✓ Using manual upload directory: {base_dir}")

# Example training command - adjust as needed
!python fnnoperator/train_heat_operator.py \
    --data-dir {data_path} \
    --dimension 2 \
    --predict-all-time-steps \
    --batch-size 4 \
    --gradient-accumulation-steps 4 \
    --use-amp \
    --n-epochs 50 \
    --learning-rate 1e-3


## Step 7: Download Results

After training, download your model and results:


In [ ]:
from google.colab import files
import glob

# Find the latest checkpoint
checkpoints = glob.glob("fnnoperator/outputs/*/checkpoints/final_model.pt")
if checkpoints:
    latest = max(checkpoints, key=os.path.getctime)
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No checkpoints found")
